### Tools
LLMs can ask to use tools for tasks like getting data from APIs, searching online, querying databases, or executing code. A tool usually consists of two parts:

1. A definition (schema) that explains the tool — including its name, purpose, and expected inputs
2. The actual function or async function that runs when the tool is called

This helps the model interact with external systems and perform real-world actions. ([LangChain Docs][1])

[1]: https://docs.langchain.com/oss/python/langchain/tools "Tools - Docs by LangChain"


In [10]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# api_key = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
# model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke("What is apple?")
response

AIMessage(content='<think>\nOkay, the user asked, What is Apple? First, I need to determine if they are referring to the company or the fruit. Since there\'s a company named Apple Inc., it\'s possible they might mean that, but I should also consider if they actually meant the fruit. However, given that Apple Inc. is a well-known company, it\'s more likely they\'re asking about the company. \n\nNext, I should outline the key points about Apple Inc. to provide a comprehensive answer. Start with the basics: founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne. Then mention the headquarters in Cupertino, California. Highlight the product lines like Mac computers, iPhones, iPads, Apple Watch, Apple TV, and services such as iTunes, Apple Music, iCloud, Apple Pay. Also, include their software ecosystem (macOS, iOS, watchOS, tvOS), and note their role in the tech industry, innovation in design, and user experience.\n\nI should also mention their impact on technology, such as the iPho

In [11]:
from langchain.tools import tool

@tool
def get_population(location:str)->str:
    """Get the current population in a given location"""
    populations = {
        "delhi": "33 million",
        "mumbai": "21 million",
        "new york": "8.5 million"
    }
    return populations.get(
        location.lower(),
        f"Population data for {location} not found."
    )


model_with_tools=model.bind_tools([get_population])

In [13]:
response = model_with_tools.invoke("What's the population in new york ?")

print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the population in New York. I need to use the get_population function. The function requires a location parameter. New York is a location, so I should call the function with "New York" as the argument. Let me make sure there\'s no ambiguity. New York could refer to the city or the state, but the function probably handles that. I\'ll proceed with "New York" as the location.\n', 'tool_calls': [{'id': 't8pavnagy', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_population'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 156, 'total_tokens': 270, 'completion_time': 0.189731446, 'completion_tokens_details': {'reasoning_tokens': 89}, 'prompt_time': 0.006575608, 'prompt_tokens_details': None, 'queue_time': 0.202412631, 'total_time': 0.196307054}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the population in New York. Let me check the tools available. There\'s a function called get_population that takes a location parameter. The required parameter is location, and it\'s a string. So I need to call get_population with "New York" as the location. I should make sure the arguments are correctly formatted as JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': '41rrxtjf4', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_population'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 156, 'total_tokens': 262, 'completion_time': 0.149772224, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.018933333, 'prompt_tokens_details': None, 'queue_time': 0.1449523, 'total_time': 0.168705557}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e0d85-0e1a-77f0-abe3-6b804344a1a6-0' tool_calls=[{'name': 'get_population', 'args': {'location': 'New York'}, 'id': '41rrxtjf4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 156, 'output_tokens': 106, 'total_tokens': 262, 'output_token_details': {'reasoning': 81}}

In [15]:
output=response.additional_kwargs['reasoning_content']
for i in output:
    if '.' in i:
        print('\n')
    print(i,end='')

Okay, the user is asking for the population in New York

. I need to use the get_population function

. The function requires a location parameter

. New York is a location, so I should call the function with "New York" as the argument

. Let me make sure there's no ambiguity

. New York could refer to the city or the state, but the function probably handles that

. I'll proceed with "New York" as the location

.


---

## Smart Restaurant Analogy 🍽️

### Step 1: Customer asks for something

```python
ai_msg = model_with_tools.invoke(messages)
```

Customer says:

> “Can I get a cold water bottle?”

Now the waiter (LLM) thinks:

* This does NOT require the kitchen
* Kitchen is for cooking meals
* Water bottle is already available in the fridge

So waiter decides:

> “Fridge is the correct place/tool.”

This is exactly how tool selection works in AI agents.

---

### Step 2: Correct tool performs the task

```python
tool_result = fridge.invoke(request)
```

The waiter gets the bottle from the fridge.

No unnecessary kitchen usage.

This represents:

* selecting the right tool
* avoiding unnecessary processing
* efficient execution

---

### Step 3: Final response to customer

```python
final_response = model_with_tools.invoke(messages)
```

Waiter returns:

> “Here is your cold water bottle.”

The LLM now formats the final human-friendly response.

---

### Mapping to AI Concepts 🤖

| Restaurant World            | AI World                 |
| --------------------------- | ------------------------ |
| Customer                    | User                     |
| Waiter                      | LLM                      |
| Fridge                      | Correct tool/API         |
| Kitchen                     | Another tool             |
| Waiter deciding where to go | Tool selection/reasoning |
| Delivering water            | Final response           |

---

### Core Idea

The important learning here is:

```text
LLM does not just call tools randomly.
It decides WHICH tool is most suitable for the task.
```

Example:

| User Request     | Correct Tool    |
| ---------------- | --------------- |
| Weather info     | Weather API     |
| Math calculation | Calculator tool |
| Database query   | SQL tool        |
| Web search       | Search API      |

This is the actual intelligence behind tool calling. ([langchain.com][1])

[1]: https://www.langchain.com/blog/tool-calling-with-langchain?utm_source=chatgpt.com "Tool Calling with LangChain"


### Tool Execution Loops

In [18]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the population in new york ?"}]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# ai_msg
# # Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_population.invoke(tool_call)
    messages.append(tool_result)
# tool_result
# # # Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The population of New York is approximately **8.5 million**.


In [19]:
messages

[{'role': 'user', 'content': "What's the population in new york ?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the population in New York. Let me check the tools available. There\'s a function called get_population that takes a location parameter. The required parameter is location, and it\'s a string. So I need to call get_population with "New York" as the location. I should make sure the arguments are correctly formatted as JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'gc25qkbxk', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_population'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 156, 'total_tokens': 262, 'completion_time': 0.150190078, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.006941353, 'prompt_tokens_details': None, 'queue_time': 0.075665474, 'total_time': 0.157131431}, 'model_name': 'qwen/qw

In [23]:
messages[1].tool_calls

[{'name': 'get_population',
  'args': {'location': 'New York'},
  'id': 'gc25qkbxk',
  'type': 'tool_call'}]

[{'role': 'user', 'content': "What's the population in new york ?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the population in New York. Let me check the tools available. There\'s a function called get_population that takes a location parameter. The required parameter is location, and it\'s a string. So I need to call get_population with "New York" as the location. I should make sure the arguments are correctly formatted as JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'gc25qkbxk', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_population'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 156, 'total_tokens': 262, 'completion_time': 0.150190078, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.006941353, 'prompt_tokens_details': None, 'queue_time': 0.075665474, 'total_time': 0.157131431}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e0d8c-b0a8-76f2-a15b-5ffc4b3d27ca-0', tool_calls=[{'name': 'get_population', 'args': {'location': 'New York'}, 'id': 'gc25qkbxk', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 106, 'total_tokens': 262, 'output_token_details': {'reasoning': 81}}),
 ToolMessage(content='8.5 million', name='get_population', tool_call_id='gc25qkbxk')]